# 1 — Dataset preparation

- Everything from raw datasets to the `.h5` files.

- Three datasets

| Dataset | Role | Size | How obtained |
|---      |---   |---   |---           |
| MIMIC-CXR | **training** — image/report pairs, no labels | 574 GB | PhysioNet credentialed |
| CheXpert (valid) | **checkpoint selection** — radiologist labels | 13 GB | Stanford AIMI / Kaggle |
| ChestX-ray14 | **test** — zero-shot evaluation | 43 GB | public (nihcc.box.com) |

- Locations come from `config/datasets.yaml`.
- Override the root with `export CXR_DATA_ROOT=/some/other/place`. 
- Only ChestX-ray14 can be downloaded by script the other two need credentials.

In [1]:
import os, sys
ROOT = "/mnt/My_Doc/github/xray_clip"            # notebooks live in notebooks/; paths below are relative to the repo root
os.chdir(ROOT); sys.path.insert(0, ROOT)

import os, subprocess, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import utils.dataset

CFG = "config/datasets.yaml"
P   = utils.dataset.load_dataset_config(CFG)

for k, v in P.items():
    print(f"  {k:<14} {v}")

  root           /mnt/My_Doc/dataset/CXR_dataset
  mimic_jpg      /mnt/My_Doc/dataset/CXR_dataset/MIMIC_CXR/mimic-cxr-jpg/2.1.0/files
  mimic_reports  /mnt/My_Doc/dataset/CXR_dataset/MIMIC_CXR/mimic-cxr-reports/files
  mimic_meta     /mnt/My_Doc/dataset/CXR_dataset/MIMIC_CXR/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-metadata.csv.gz
  mimic_split    /mnt/My_Doc/dataset/CXR_dataset/MIMIC_CXR/mimic-cxr-jpg/2.1.0/mimic-cxr-2.0.0-split.csv.gz
  chex_val_dir   /mnt/My_Doc/dataset/CXR_dataset/CheXpert/kaggle/valid
  chex_val_csv   /mnt/My_Doc/dataset/CXR_dataset/CheXpert/kaggle/valid.csv
  chex_test_csv  /mnt/My_Doc/dataset/CXR_dataset/CheXpert/test_set_labels/groundtruth.csv
  nih_images     /mnt/My_Doc/dataset/CXR_dataset/ChestXray14/images
  nih_entry      /mnt/My_Doc/dataset/CXR_dataset/ChestXray14/cxr8/Data_Entry_2017_v2020.csv
  nih_test       /mnt/My_Doc/dataset/CXR_dataset/ChestXray14/cxr8/test_list.txt
  nih_trainval   /mnt/My_Doc/dataset/CXR_dataset/ChestXray14/cxr8/train_val_list.txt
  d

## 1. Availability

- `check_datasets()` reports
    - which raw files are present, 
    - summarises each dataset, 
    - and checks the derived artefacts in `CheXzero/data/`. 
- Anything missing prints where to obtain it.

In [ ]:
import utils.dataset

CFG = "config/datasets.yaml"
utils.dataset.check_datasets(cfg_path = CFG)

## 2. MIMIC-CXR — how it is laid out

`files/pXX/pXXXXXXXX/sXXXXXXXX/<dicom_id>.jpg`, where `pXX` is a 10-way shard of the patient
id. One **study** (`sXXXXXXXX`) holds one or more images and has exactly one free-text report
at the matching path under `mimic-cxr-reports/`.

That one-report-per-study structure is the whole training signal: every image in a study is
paired with the same report text.

In [ ]:
pat = sorted(P["mimic_jpg"].glob("p10/p10000032"))[0]
print(f"patient dir: {pat}\n")
for study in sorted(pat.iterdir()):
    imgs = sorted(study.glob("*.jpg"))
    print(f"  {study.name}/   {len(imgs)} image(s)")
    for im in imgs:
        print(f"      {im.name}   {im.stat().st_size/1e6:.1f} MB")

rep = P["mimic_reports"]/pat.parent.name/pat.name/f"{sorted(pat.iterdir())[0].name}.txt"
print(f"\nmatching report: {rep}")

In [ ]:
# One study, its images, and the report they all share.
study = sorted(pat.iterdir())[0]
imgs  = sorted(study.glob("*.jpg"))
meta  = pd.read_csv(P["mimic_meta"])
vp    = meta.set_index("dicom_id")["ViewPosition"]

fig, axes = plt.subplots(1, len(imgs), figsize=(5*len(imgs), 5.5))
axes = np.atleast_1d(axes)
for ax, im in zip(axes, imgs):
    ax.imshow(plt.imread(im), cmap="gray")
    ax.set_title(f"{vp.get(im.stem, '?')}\n{im.stem[:18]}...", fontsize=10)
    ax.axis("off")
plt.suptitle(f"study {study.name} — one report, {len(imgs)} images", fontsize=12)
plt.tight_layout(); plt.show()

txt = (P["mimic_reports"]/pat.parent.name/pat.name/f"{study.name}.txt").read_text()
print("REPORT (this text pairs with EVERY image above):\n")
print(textwrap.indent(txt.strip(), "  "))

### 2b. Report sections

A full report has EXAMINATION / INDICATION / TECHNIQUE / COMPARISON / FINDINGS / IMPRESSION.
Only **IMPRESSION** (with FINDINGS as fallback) is extracted for training — the header is
near-identical across reports, so including it would give many images near-identical text and
make the contrastive task unsolvable for those rows.

CLIP's context is **77 tokens**, so section choice is also a budget decision.

In [ ]:
import re
def sections(txt):
    out, cur = {}, None
    for line in txt.splitlines():
        m = re.match(r"\s*([A-Z][A-Z /()]+):\s*(.*)", line)
        if m:
            cur = m.group(1).strip(); out[cur] = m.group(2).strip()
        elif cur:
            out[cur] = (out[cur] + " " + line.strip()).strip()
    return out

s = sections(txt)
print(f"{'section':<28}{'words':>7}   first 60 chars")
print("-"*100)
for k, v in s.items():
    print(f"{k:<28}{len(v.split()):>7}   {v[:60]}")
print("\n-> training uses IMPRESSION only (FINDINGS as fallback when absent)")

### 2c. View position — the train/test mismatch

The CheXzero paper says: *"the chest X-ray that is in anteroposterior/posteroanterior view was
chosen to be included as part of training."* The released code does **not** filter — it globs
every `.jpg`. Our run followed the code.

ChestX-ray14 is 100% frontal, so any non-frontal training image is a view the model never sees
at test time. `ViewPosition` comes from `mimic-cxr-2.0.0-metadata.csv.gz`; the JPG filename is
the `dicom_id`, so it joins directly onto the image paths.

In [ ]:
front = meta["ViewPosition"].isin(["AP", "PA"])
print(f"{'view':<12}{'images':>10}{'%':>8}")
print("-"*30)
for k, n in meta["ViewPosition"].fillna("(missing)").value_counts().head(6).items():
    print(f"{k:<12}{n:>10,}{100*n/len(meta):>7.1f}%")
print("-"*30)
print(f"{'FRONTAL':<12}{front.sum():>10,}{100*front.mean():>7.1f}%")
print(f"{'non-frontal':<12}{(~front).sum():>10,}{100*(~front).mean():>7.1f}%")

# what a lateral actually looks like next to a frontal
lat = meta[meta.ViewPosition == "LATERAL"].iloc[0]
ap  = meta[meta.ViewPosition == "AP"].iloc[0]
fig, ax = plt.subplots(1, 2, figsize=(9, 5))
for a, row, ttl in zip(ax, [ap, lat], ["AP (frontal)", "LATERAL"]):
    p = (P["mimic_jpg"]/f"p{str(row.subject_id)[:2]}"/f"p{row.subject_id}"
         /f"s{row.study_id}"/f"{row.dicom_id}.jpg")
    a.imshow(plt.imread(p), cmap="gray"); a.set_title(ttl, fontsize=11); a.axis("off")
plt.tight_layout(); plt.show()

## 3. CheXpert validation — checkpoint selection

234 images / 200 patients, labelled by three radiologists. Used **only** to rank training
checkpoints by mean AUC over the five competition tasks (Atelectasis, Cardiomegaly,
Consolidation, Edema, Pleural Effusion). Never used for training, never reported as a result.

Preprocessing keeps the `view1` frontals — exactly 200, one per patient.

In [ ]:
v = pd.read_csv(P["chex_val_csv"])
print(f"{len(v)} rows   columns: {list(v.columns[:6])} ...\n")
print(v[["Path", "Sex", "Age", "Frontal/Lateral", "AP/PA"]].head(3).to_string(index=False))

comp = ["Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Pleural Effusion"]
print(f"\n{'task':<20}{'positives':>10}{'prevalence':>12}")
print("-"*42)
for c in comp:
    print(f"{c:<20}{int(v[c].sum()):>10}{100*v[c].mean():>11.1f}%")

sel = v[v.Path.str.contains("view1")].head(4)
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for a, (_, r) in zip(ax, sel.iterrows()):
    p = P["chex_val_dir"].parent/r.Path.replace("CheXpert-v1.0-small/", "")
    a.imshow(plt.imread(p), cmap="gray"); a.axis("off")
    pos = [c.split()[-1][:6] for c in comp if r[c] == 1] or ["(none)"]
    a.set_title(f"{r['AP/PA']}  {', '.join(pos)}", fontsize=9)
plt.tight_layout(); plt.show()

## 4. ChestX-ray14 — the test set

112,120 PNGs, 1024x1024, all frontal (PA 60% / AP 40%). Labels are multi-hot in
`Data_Entry_2017_v2020.csv`; 54% are "No Finding".

Two splits matter:
* `test_list.txt` — 25,596 images, the official benchmark
* `train_val_list.txt` — 86,524 images, **disjoint from test**, used for prompt tuning

In [ ]:
d  = pd.read_csv(P["nih_entry"])
te = set(P["nih_test"].read_text().split())
tv = set(P["nih_trainval"].read_text().split())
print(f"images {len(d):,}   test {len(te):,}   train/val {len(tv):,}   overlap {len(te & tv)}")

lab = sorted({x for s in d["Finding Labels"] for x in s.split("|")} - {"No Finding"})
cnt = {l: d["Finding Labels"].str.contains(l, regex=False).sum() for l in lab}
print(f"\n{'pathology':<20}{'images':>9}{'%':>8}")
print("-"*38)
for l, n in sorted(cnt.items(), key=lambda x: -x[1]):
    print(f"{l:<20}{n:>9,}{100*n/len(d):>7.1f}%")

sel = d[d["Finding Labels"] != "No Finding"].head(4)
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for a, (_, r) in zip(ax, sel.iterrows()):
    a.imshow(plt.imread(P["nih_images"]/r["Image Index"]), cmap="gray"); a.axis("off")
    a.set_title(f"{r['View Position']}  {r['Finding Labels'][:28]}", fontsize=9)
plt.tight_layout(); plt.show()

## 5. Building the `.h5` files

Training reads a single HDF5 array, not 377k JPEGs. `run_preprocess.py` walks the images,
records the order in a paths CSV, and writes row *i* of the h5 from path *i*.

| Output | Contents |
|---|---|
| `data/cxr_paths.csv` | the canonical ordering — 377,110 paths |
| `data/cxr.h5` | `(377110, 320, 320)` uint8, ~38.6 GB |
| `data/mimic_impressions.csv` | 377,110 rows, one per **image** |

**Row *i* of the h5 and row *i* of the CSV are the same pair.** That index correspondence is
the entire contrastive signal, so the two must always be regenerated together.

Why uint8: `preprocess()` returns an 8-bit image, so h5py's float32 default stores the same
values in 4x the space. Why 320: images are letterboxed to square at 320, then resized to 224
at load time for ViT-B/32.

⚠️ `--radiology_reports_path` **must** end in a trailing slash — the report path is built by
string concatenation.

In [ ]:
# MIMIC: images -> cxr.h5 AND reports -> mimic_impressions.csv, one pass. ~3 h (HDD-bound).
# Skip if CheXzero/data/cxr.h5 already exists.
cmd = f"""cd CheXzero && python -u run_preprocess.py \\
    --dataset_type mimic \\
    --chest_x_ray_path {P['mimic_jpg']} \\
    --radiology_reports_path {P['mimic_reports']}/ \\
    --csv_out_path data/cxr_paths.csv \\
    --cxr_out_path data/cxr.h5 \\
    --mimic_impressions_path data/mimic_impressions.csv"""
print(cmd)
# !{cmd}

In [ ]:
# Evaluation sets — minutes, not hours.
for name, args in {
    "CheXpert valid (selection)":
        f"--dataset_type chexpert-valid --chest_x_ray_path {P['chex_val_dir']} "
        f"--csv_out_path data/chexpert_valid_paths.csv --cxr_out_path data/chexpert_valid.h5",
    "ChestX-ray14 test (25,596)":
        f"--dataset_type nih-test --chest_x_ray_path {P['nih_images']} "
        f"--image_list {P['nih_test']} "
        f"--csv_out_path data/nih_test_paths.csv --cxr_out_path data/nih_test.h5",
}.items():
    print(f"# {name}")
    print(f"cd CheXzero && python -u run_preprocess.py {args}\n")

## 5b. Frontal-only subset

The CheXzero paper says *"the chest X-ray that is in anteroposterior/posteroanterior view was
chosen to be included as part of training"* — but the released code never filters, so our
`cxr.h5` contains all 377,110 images, 35.5% of them non-frontal. ChestX-ray14 is 100% frontal,
so over a third of training was spent on a view the model never sees at test time.

`ViewPosition` leaves 15,769 images (4.2%) unlabelled. Two subgroups are recoverable from
metadata alone, without looking at pixels:

| Group | Count | Why it must be frontal |
|---|---|---|
| strict `AP`/`PA` | 243,334 | labelled directly |
| **B** — "PA AND LAT" ordered, sibling labelled `LL` | 2,239 | the `LL` sibling *is* the lateral, so this is the frontal |
| **C** — "CHEST (PORTABLE AP)" | 2,215 | portable chest films are AP by definition |
| **total kept** | **247,788** | |

The remaining 129,322 are laterals plus 11,219 genuinely-ambiguous images.

No re-decoding is needed: the JPG filename **is** the `dicom_id`, so we build a row mask and
copy those rows straight out of the existing `cxr.h5`. The same mask is applied to the
impressions CSV, which is what preserves the row-*i*-to-row-*i* alignment that training depends on.


In [4]:
import h5py, numpy as np, pandas as pd

D          = P["derived"]
SRC_H5     = D/"cxr.h5"
SRC_TXT    = D/"mimic_impressions_v2.csv"      # impression + FINDINGS fallback
OUT_H5     = D/"cxr_frontal.h5"
OUT_TXT    = D/"mimic_impressions_frontal.csv"
RECOVER    = True                               # False -> strict AP/PA only (243,334)

# ── build the mask, in cxr_paths.csv row order ──────────────────────────────────
meta  = pd.read_csv(P["mimic_meta"])
known = (meta[meta.ViewPosition.notna()]
         .groupby("study_id")["ViewPosition"].apply(lambda s: "+".join(sorted(set(s)))))
meta["sibling"] = meta.study_id.map(known).fillna("none")
meta["desc"]    = meta.PerformedProcedureStepDescription.fillna("(none)")

empty = meta.ViewPosition.isna()
keep  = meta.ViewPosition.isin(["AP", "PA"])
if RECOVER:
    keep |= empty & (meta.desc == "CHEST (PA AND LAT)") & (meta.sibling == "LL")   # B
    keep |= empty & (meta.desc == "CHEST (PORTABLE AP)")                            # C
keep_ids = set(meta.loc[keep, "dicom_id"])

paths = pd.read_csv(D/"cxr_paths.csv")["Path"]
did   = paths.str.rsplit("/", n=1).str[-1].str.replace(".jpg", "", regex=False)
mask  = did.isin(keep_ids).to_numpy()
idx   = np.flatnonzero(mask)
print(f"keeping {len(idx):,} of {len(mask):,}  ({100*mask.mean():.1f}%)")

# ── subset the text first: cheap, and a mismatch here must abort before the 25 GB copy ──
txt = pd.read_csv(SRC_TXT)
assert len(txt) == len(paths), f"text {len(txt)} != paths {len(paths)}"
txt.iloc[idx].to_csv(OUT_TXT, index=False)
print(f"wrote {OUT_TXT.name}  {len(idx):,} rows")

# ── copy the h5 rows in chunks ─────────────────────────────────────────────────
with h5py.File(SRC_H5, "r") as fin, h5py.File(OUT_H5, "w") as fout:
    src = fin["cxr"]
    assert src.shape[0] == len(mask), f"h5 {src.shape[0]} != paths {len(mask)}"
    dst = fout.create_dataset("cxr", shape=(len(idx), *src.shape[1:]), dtype=src.dtype)
    CH = 2048
    for s in range(0, len(idx), CH):
        block = idx[s:s+CH]
        dst[s:s+len(block)] = src[block]        # h5py needs an increasing index list
        if (s // CH) % 20 == 0:
            print(f"  {s:>7,}/{len(idx):,}", end="\r")
print(f"\nwrote {OUT_H5.name}  {OUT_H5.stat().st_size/1e9:.1f} GB")

# ── verify ─────────────────────────────────────────────────────────────────────
with h5py.File(OUT_H5, "r") as f:
    d = f["cxr"]
    n = d.shape[0]
    print(f"\n{OUT_H5.name}: {d.shape} {d.dtype}")
    for i in [0, n//2, n-1]:
        a = d[i]
        print(f"  row {i:>7}: min={a.min():3d} max={a.max():3d} nonzero={100*(a>0).mean():5.1f}%")
    assert n == len(pd.read_csv(OUT_TXT)), "h5 and text row counts differ"
    print(f"\nALIGNED: h5 {n:,} == text {n:,}")

# spot-check that a kept row really is frontal
vp = meta.set_index("dicom_id")["ViewPosition"]
chk = did.iloc[idx[:5]].map(vp).fillna("(empty->recovered)")
print("\nfirst 5 kept rows:", list(chk))


keeping 247,788 of 377,110  (65.7%)
wrote mimic_impressions_frontal.csv  247,788 rows
  245,760/247,788
wrote cxr_frontal.h5  25.4 GB

cxr_frontal.h5: (247788, 320, 320) uint8
  row       0: min=  0 max=255 nonzero= 80.6%
  row  123894: min=  0 max=255 nonzero= 85.3%
  row  247787: min=  0 max=255 nonzero= 82.3%

ALIGNED: h5 247,788 == text 247,788

first 5 kept rows: ['AP', 'PA', 'AP', 'AP', 'PA']


## 6. Verify before training

Three things must hold, and a silent failure in any of them trains the model on garbage while
the loss still looks plausible:

1. h5 rows == impressions rows == paths rows
2. pixels span 0–255 and no row is all-zero (a truncated write)
3. the path manifests still resolve — needed only to *rebuild*, but a stale manifest means the
   next preprocessing run fails on every path

In [ ]:
import h5py
D = P["derived"]

with h5py.File(D/"cxr.h5", "r") as f:
    dset = f["cxr"]; n_img = dset.shape[0]
    print(f"cxr.h5          {dset.shape}  {dset.dtype}  {dset.nbytes/1e9:.1f} GB")
    for i in [0, n_img//2, n_img-1]:
        a = dset[i]
        print(f"  row {i:>7}: min={a.min():3d} max={a.max():3d} nonzero={100*(a>0).mean():5.1f}%")
    blank = sum(1 for i in np.linspace(0, n_img-1, 200, dtype=int) if dset[i].max() == 0)
    print(f"  all-zero rows in 200-row sample: {blank}")

imp   = pd.read_csv(D/"mimic_impressions.csv")
paths = pd.read_csv(D/"cxr_paths.csv")
print(f"\nimpressions {len(imp):,}   paths {len(paths):,}   h5 {n_img:,}")
print(f"ALIGNED: {len(imp) == len(paths) == n_img}")

noimp = (imp['impression'] == 'NO IMPRESSION').mean()
print(f"\n'NO IMPRESSION': {100*noimp:.1f}%   NaN: {imp['impression'].isna().sum()}")
print(f"words: median {imp['impression'].astype(str).str.split().str.len().median():.0f}")

## Next

`2_main.ipynb` — training, checkpoint selection on CheXpert, prompt tuning on held-out
ChestX-ray14 train/val, and zero-shot evaluation.

# checking how many times disease is called in the report